In [1]:
import pickle
import mediapipe as mp
import cv2
import numpy as np

In [4]:
#cargar el modelo de clasificaion 
model_dict = pickle.load(open('./asl_model01.p', 'rb'))
model = model_dict['model']

#configurar la camara
cap = cv2.VideoCapture(0)

# configurar mediapipe handlandmarker
BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)


#crear el detector
with HandLandmarker.create_from_options(options) as hands:

    frame_timestamp_ms = 33

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        H, W, _ = frame.shape

        # convertir los canales de color 'bgr' a 'rgb
        frame_rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # convertir a MediaPipe Image
        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame_rgb
        )

        # marca de tiempo obligatorio para video
        frame_timestamp_ms += 33

        # Detectar manos
        results = hands.detect_for_video(
            mp_image,
            frame_timestamp_ms
        )

        # --------------------------------------------------
        # Procesar landmarks
        # --------------------------------------------------

        if results.hand_landmarks:

            for hand_landmarks in results.hand_landmarks:

                data_aux = []

                x_ = [lm.x for lm in hand_landmarks]
                y_ = [lm.y for lm in hand_landmarks]

                for lm in hand_landmarks:

                    data_aux.append(
                        lm.x - min(x_)
                    )

                    data_aux.append(
                        lm.y - min(y_)
                    )

                # --------------------------------------------------
                # Predicción
                # --------------------------------------------------

                prediction = model.predict(
                    [np.asarray(data_aux)]
                )

                predicted_character = prediction[0]

                cv2.putText(
                    frame,
                    predicted_character,
                    (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    1.3,
                    (0, 255, 0),
                    3
                )

        # Mostrar cámara
        cv2.imshow(
            'ASL-Vision-12',
            frame
        )

        # Presionar Q para salir
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break


# --------------------------------------------------
# Liberar recursos
# --------------------------------------------------

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1787094580.710965 1555719 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1787094580.719218 1555735 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.1.6), renderer: Mesa Intel(R) UHD Graphics 630 (CFL GT2)
W0000 00:00:1787094580.788845 1555722 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787094580.840640 1555729 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1787094582.354211 1555723 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.
QFontDatabase: Cannot find font directory /home/fred/miniconda3/envs/Deus/lib/python3.10/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (fr